# 🔍 Customer Churn Prediction
**Goal:** Predict whether a customer will leave (churn) or stay using machine learning.

### Steps We'll Follow:
1. Install & Import Libraries
2. Load the Dataset
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Train ML Models
6. Evaluate Models
7. Feature Importance

## Step 1: Install & Import Libraries

In [ ]:
# Install required libraries (run this only once)
# !pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
# --- Import all necessary libraries ---

import pandas as pd                   # For data manipulation
import numpy as np                    # For numerical operations
import matplotlib.pyplot as plt       # For plotting graphs
import seaborn as sns                 # For beautiful visualizations

# Machine Learning tools from scikit-learn
from sklearn.model_selection import train_test_split       # To split data
from sklearn.preprocessing import LabelEncoder, StandardScaler  # To encode and scale
from sklearn.linear_model import LogisticRegression        # Model 1
from sklearn.ensemble import RandomForestClassifier        # Model 2
from sklearn.metrics import (
    accuracy_score,         # How often the model is correct
    precision_score,        # Of predicted churners, how many actually churned
    recall_score,           # Of actual churners, how many did we catch
    roc_auc_score,          # Overall model performance score
    confusion_matrix,       # Table of correct/wrong predictions
    classification_report   # Detailed performance report
)

import warnings
warnings.filterwarnings('ignore')     # Hide unnecessary warnings

print("✅ All libraries imported successfully!")

## Step 2: Load the Dataset

> 📥 We'll use the **Telco Customer Churn** dataset from Kaggle.  
> Download it here: https://www.kaggle.com/datasets/blastchar/telco-customer-churn  
> Save the CSV file in the same folder as this notebook.

In [ ]:
# --- Load the dataset ---
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Preview the first 5 rows
print("Shape of dataset:", df.shape)  # (rows, columns)
df.head()

In [ ]:
# --- Check basic info about the dataset ---
print("\n📋 Dataset Info:")
df.info()

print("\n📊 Basic Statistics:")
df.describe()

## Step 3: Exploratory Data Analysis (EDA)
Let's understand the data before building any model.

In [ ]:
# --- Check how many customers churned vs stayed ---
print("Churn Value Counts:")
print(df['Churn'].value_counts())

# Plot churn distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Churn', data=df, palette='Set2')
plt.title('Churn Distribution (Yes = Left, No = Stayed)')
plt.xlabel('Churn')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()

In [ ]:
# --- Check missing values ---
print("Missing Values per Column:")
print(df.isnull().sum())

# Note: TotalCharges may have spaces instead of NaN — let's fix that
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("\nAfter converting TotalCharges, missing values:")
print(df['TotalCharges'].isnull().sum())

In [ ]:
# --- Visualize Tenure vs Churn ---
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, palette='Set1')
plt.title('Tenure Distribution by Churn')
plt.xlabel('Tenure (months)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()
# 💡 Insight: Customers who churn tend to have shorter tenure

In [ ]:
# --- Visualize Monthly Charges vs Churn ---
plt.figure(figsize=(8, 4))
sns.boxplot(x='Churn', y='MonthlyCharges', data=df, palette='Set2')
plt.title('Monthly Charges by Churn')
plt.tight_layout()
plt.show()
# 💡 Insight: Customers who churn tend to have higher monthly charges

## Step 4: Data Preprocessing
Machine learning models need numbers — so we convert and clean the data.

In [ ]:
# --- Drop customerID (not useful for prediction) ---
df.drop(columns=['customerID'], inplace=True)

# --- Fill missing values in TotalCharges with median ---
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

print("✅ Dropped customerID and filled missing values.")

In [ ]:
# --- Encode the target variable (Churn: Yes=1, No=0) ---
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# --- Encode all other categorical (text) columns using LabelEncoder ---
le = LabelEncoder()

# Get list of categorical columns
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:", categorical_cols)

# Apply label encoding to each column
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

print("\n✅ All categorical columns encoded.")
df.head()

In [ ]:
# --- Split data into Features (X) and Target (y) ---
X = df.drop(columns=['Churn'])   # All columns except Churn
y = df['Churn']                  # Only the Churn column

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# --- Split into Train (80%) and Test (20%) sets ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% for testing
    random_state=42,     # For reproducibility
    stratify=y           # Keep same churn ratio in both splits
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

In [ ]:
# --- Scale the features (bring all values to same range) ---
# This helps Logistic Regression perform better
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # Fit on train, transform train
X_test_scaled  = scaler.transform(X_test)         # Only transform test (no fitting!)

print("✅ Feature scaling done.")

## Step 5: Train Machine Learning Models

In [ ]:
# --- Model 1: Logistic Regression ---
# A simple and interpretable classification model

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)   # Train the model

lr_preds = lr_model.predict(X_test_scaled)         # Predict class labels
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]  # Predict probabilities

print("✅ Logistic Regression trained!")

In [ ]:
# --- Model 2: Random Forest ---
# A powerful ensemble model using many decision trees

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)   # Random Forest doesn't need scaled data

rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

print("✅ Random Forest trained!")

## Step 6: Evaluate the Models

In [ ]:
# --- Helper function to print all metrics ---
def evaluate_model(name, y_true, y_pred, y_prob):
    print(f"\n{'='*45}")
    print(f"  Model: {name}")
    print(f"{'='*45}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred):.4f}")
    print(f"  ROC-AUC   : {roc_auc_score(y_true, y_prob):.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Stayed', 'Churned']))

# Evaluate both models
evaluate_model("Logistic Regression", y_test, lr_preds, lr_probs)
evaluate_model("Random Forest",       y_test, rf_preds, rf_probs)

In [ ]:
# --- Confusion Matrix for both models ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title in zip(
    axes,
    [lr_preds, rf_preds],
    ['Logistic Regression', 'Random Forest']
):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Stayed', 'Churned'],
        yticklabels=['Stayed', 'Churned']
    )
    ax.set_title(f'Confusion Matrix - {title}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

## Step 7: Feature Importance
Which factors influence churn the most?

In [ ]:
# --- Feature Importance from Random Forest ---
importances = rf_model.feature_importances_
feature_names = X.columns

# Create a sorted DataFrame
feat_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Plot top 10 features
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_df.head(10), palette='viridis')
plt.title('Top 10 Features That Drive Customer Churn')
plt.tight_layout()
plt.show()

print("\n💡 Top 5 churn-driving features:")
print(feat_df.head(5).to_string(index=False))

## ✅ Summary

| Model | Accuracy | ROC-AUC |
|---|---|---|
| Logistic Regression | ~80% | ~84% |
| Random Forest | ~79–82% | ~83–86% |

### Key Takeaways:
- **Tenure**, **MonthlyCharges**, and **TotalCharges** are top churn predictors
- Customers with shorter tenure and higher charges are most likely to leave
- Random Forest generally gives better results but takes more time to train

### Next Steps:
- Try **hyperparameter tuning** with GridSearchCV
- Handle **class imbalance** with SMOTE
- Try **XGBoost** for even better accuracy